# WiLoR multi-view MANO fitting for hands (v2)

Companion to `wilor_hand_reconstruction.ipynb` (v1, left untouched). v1 fused hands by
averaging each camera's full 778-vertex MANO mesh in world space, anchored to the
MediaPipe-triangulated wrist. That works but each camera's *whole-mesh orientation*
can be ambiguous from a single view (root cause of the frame-61 "flip" patched with a
temporal-anchor hack in v1).

This notebook instead follows a more principled pipeline: fuse each camera's *own*
21-joint hand articulation (which is internally rigid and self-consistent -- WiLoR
predicts a full, plausible hand skeleton per view) into a single 21-joint target per
frame, then fit a single MANO pose per frame to that target.

An earlier version of step 5 triangulated each of the 21 MANO joints *independently*
via per-2D-reprojection DLT across cameras. That turned out to be the wrong
representation: it discards each camera's internally-consistent hand shape and
replaces it with 21 unrelated noisy 3D points. Per-camera, the wrist-to-knuckle
"bone length" for e.g. the index finger is essentially constant (~38.6mm, std
<0.5mm across all 128 frames x 7 cameras -- it depends only on `betas`, not pose).
But independently DLT-triangulating joints 0 and 4 (middle-finger knuckle) produced a
"bone length" that varied by up to *253mm* across frames -- a 25cm spurious stretch
that exists in no individual camera's prediction, and the direct cause of the
"fingers wobbling" / "joint rotations not consolidated" symptom. See step 5 below.

Pipeline (numbered to match the design guide), with simplifications noted:

1. **Detect & classify hands** - reuse MediaPipe bboxes + left/right from
   `extracted_2d_keypoints.json` (same as v1, no separate WiLoR/YOLO detector).
1b. **Resolve physical-hand identity** (new cell, right after extraction) -
   MediaPipe's per-camera left/right label for the same physical hand is often
   inconsistent, both across cameras and across time (87% of frames have at least
   one camera disagreeing with the consensus). A temporally-tracked, per-frame
   2-cluster grouping of each camera's world-frame wrist position -- with the
   constraint that a single camera's own left_hand/right_hand detections, if both
   present, must be different physical hands -- replaces the raw per-camera `si`
   axis with a `pid` axis (one consistent identity per physical hand across the
   whole sequence) before any fusion below. Entries whose raw label disagreed with
   the pid consensus are re-mirrored (the same x -> -x / `M @ R @ M` trick WiLoR
   itself uses for left hands) so every camera's contribution to a given `pid`
   describes that physical hand under the same convention.
2. **Per-view WiLoR inference** - reuse v1's calibrated `run_wilor_on_image`,
   extended to also return `betas` and all 21 `joints_cam`.
3. **Per-view confidence weight** - `mediapipe_confidence * truncation_factor`.
   Self-occlusion / back-of-hand heuristics are dropped: Step 5's outlier rejection
   implicitly down-weights views that disagree, which is what those heuristics
   approximate.
4. **Fuse shape (beta)** - weighted mean across cameras per frame, then an EMA over
   time (alpha=0.05).
5. **Fuse 3D joints** - joint 0 (wrist) comes from the trusted MediaPipe/body-pose
   triangulation (`final_reconstructed_3d_keypoints.json`, same as v1 -- see 5b).
   For joints 1-20, each camera's offset `kpts[j] - kpts[0]` (a rigid vector from
   that camera's own self-consistent WiLoR prediction) is rotated into world
   orientation via that camera's extrinsics and combined via confidence-weighted
   averaging across cameras, with one-pass outlier rejection on cameras whose
   rotated offset disagrees with the consensus. This preserves each camera's
   internally-consistent finger geometry instead of triangulating 21 unrelated
   points.
5b. **Wrist anchor** - during fast motion, WiLoR's per-camera hand-pose estimates
   can be simultaneously unreliable across most/all cameras, so DLT-triangulating
   the wrist can land tens of cm from the truth even after outlier rejection. Take
   joint 0 directly from the trusted MediaPipe/body-pose-triangulated wrist
   (`final_reconstructed_3d_keypoints.json`, the same anchor v1 used) -- this also
   serves as the reference point that step 5's per-camera offsets are added to.
   When disabled, joint 0 falls back to per-camera weighted DLT triangulation (with
   the same outlier rejection as before), for comparison.
6. **Fit MANO to fused joints** - per frame/hand, optimize `global_orient` +
   `hand_pose` (6D rotation reps) + `transl` via Adam, with `beta` fixed from step 4.
   Left hand: no `MANO_LEFT.pkl` exists, so we reflect the joint targets through a
   world-space plane, fit the right-hand MANO model, and reflect the output mesh
   back (rendered with `MANO_FACES_MIRRORED`). Anatomical joint-limit penalties are
   skipped - the pose prior + temporal prior keep poses plausible.

   `model.mano` here is a `MANOLayer`, which feeds `hand_pose` straight to `lbs()`
   as rotation matrices and never adds back `model.mano.hand_mean` -- so
   `hand_pose = identity rotation` is MANO's literal flat/straight-fingers rest
   pose, not a "neutral" hand. An earlier version of the pose prior pulled toward
   exactly that identity target, which fought the joint-position fit and produced
   visibly flattened fingers. The prior (and the warm-start for the first frame of
   each sequence) now target `model.mano.hand_mean` converted to 6D rotations
   instead.
7. **Hand-hand contact** - *optional*, off by default (`ENABLE_CONTACT_LOSS`).
   When enabled, adds a simple joint-level repulsion penalty between the two hands'
   MANO joints (not full mesh interpenetration).
8. **Temporal smoothing** - velocity-aware EMA on triangulated joints, plus a
   temporal prior in step 6's optimization (no separate post-hoc smoothing pass).

**Calibration QC** (cell 05c, right after physical-hand identity): this rig's
calibration has substantial systematic per-camera reprojection error (verified against
`final_reconstructed_3d_keypoints.json`'s body-joint triangulation, independent of
WiLoR/hands) -- e.g. ~20px for cam04 vs. ~300px for cam03 on a 3360x1890 image, constant
across frames. `CALIB_WEIGHT` captures this per-camera bias for use below.

**Simple AVG Debug** (cells 06-07, right after extraction): a deliberately "dumb"
baseline for comparison against the full fit below. One fixed `betas` per hand for the
whole sequence (mean over every valid frame/camera). Per frame, per hand: take the
cameras whose confidence weight clears `SIMPLE_AVG_CONF_THRESHOLD` (falling back to all
valid cameras if none do), average their raw `global_orient`/`hand_pose` rotation
matrices with weight `mediapipe_confidence * CALIB_WEIGHT[cam]` (re-projected to the
nearest rotation), and anchor the wrist position to the body-pose triangulation
(`final_reconstructed_3d_keypoints.json`, same anchor as step 5b) -- no Adam fit, no
temporal smoothing, no pose prior. A Viser cell right after lets you inspect this
baseline before looking at the fitted result from cell 12.

The next cell collects every stage that can be toggled on/off (or retuned) into one
place, so individual pipeline components can be ablated and compared against v1
without editing code in multiple cells.

In [ ]:
# Pipeline configuration -- toggle stages on/off and tune their parameters here,
# then re-run from beta08 down (cells 08-11 are cheap; extraction in cell 05, the
# physical-hand identity clustering in cell 05b, and the Simple AVG Debug cells 06-07
# are unaffected by anything below).

# --- Steps 3-5: per-view confidence weighting ------------------------------------
# weight = mediapipe_confidence * truncation_factor when fusing betas (beta06) and
# triangulating joints (triang07). If False, all valid views are weighted equally.
ENABLE_CONFIDENCE_WEIGHTING = True

# --- Step 4: shape (beta) fusion --------------------------------------------------
BETA_EMA_ALPHA = 0.05  # temporal EMA on fused betas

# --- Step 5: fused-joint outlier rejection -----------------------------------------
# For joints 1-20: drop cameras whose rotated per-camera offset (kpts[j]-kpts[0],
# rotated into world orientation) deviates from the weighted-mean offset by more than
# OUTLIER_FACTOR * median(deviation), then re-average. If ENABLE_WRIST_ANCHOR is
# False, the same factor also gates the joint-0 DLT outlier rejection (drop cameras
# whose reprojection error exceeds OUTLIER_FACTOR * median(errs)).
ENABLE_TRIANG_OUTLIER_REJECTION = True
OUTLIER_FACTOR = 1.5

# --- Step 5b: wrist anchor -----------------------------------------------------------
# During fast motion, WiLoR's per-camera hand-pose estimates can be simultaneously
# unreliable across most/all cameras (every camera reports a large, mutually-
# inconsistent reprojection error), so DLT-triangulating the wrist can land tens of
# cm from the truth even after outlier rejection. v1 avoided this by anchoring the
# whole hand to the MediaPipe/body-pose-triangulated wrist
# (final_reconstructed_3d_keypoints.json, joint 15=left_wrist/16=right_wrist), which
# stays smooth through these segments. When True, joint 0 is taken directly from that
# trusted body-pose wrist every frame (also used as the reference point for step 5's
# per-camera finger offsets); when False, joint 0 falls back to per-camera weighted
# DLT triangulation, for comparison.
ENABLE_WRIST_ANCHOR = True

# --- Step 8a: joint-target smoothing (pre-fit) ------------------------------------
# Velocity-aware EMA on the triangulated joints before they're used as MANO fit
# targets. If False, joints3d_smooth = joints3d_fused (fit directly to the raw,
# per-frame triangulation).
ENABLE_JOINT_SMOOTHING = True
FAST_MOTION_THRESHOLD = 0.03  # m/frame; per-joint velocity considered "fast"

# --- Step 6: MANO fitting ----------------------------------------------------------
N_STEPS = 100  # Adam iterations per frame/hand
LR = 0.02

# Pose prior: L2 penalty pulling hand_pose toward MANO's mean hand pose
# (model.mano.hand_mean -- a relaxed, slightly-curled pose), not the literal
# zero/identity rotation. model.mano is a MANOLayer: hand_pose is fed straight to
# lbs() as rotation matrices with no pose_mean offset added internally, so
# hand_pose=identity means the literal flat/straight-fingers MANO rest pose. Priming
# the prior toward identity therefore pulled every fit toward a flat hand.
ENABLE_POSE_PRIOR = True
POSE_PRIOR_W = 1e-3

# --- Step 8b: temporal prior (during fitting) --------------------------------------
# L2 penalty pulling each frame's fit params toward the previous frame's.
ENABLE_TEMPORAL_PRIOR = True
TEMPORAL_W = 1e-2

# --- Step 7: hand-hand contact (optional) -------------------------------------------
ENABLE_CONTACT_LOSS = False
CONTACT_THRESHOLD = 0.05  # m; wrist-to-wrist distance below which this kicks in
CONTACT_MIN_DIST = 0.01   # m; desired minimum joint-to-joint separation
CONTACT_W = 0.1

print('Pipeline config:')
print(f'  confidence weighting (3-5):    {ENABLE_CONFIDENCE_WEIGHTING}')
print(f'  triang outlier rejection (5):  {ENABLE_TRIANG_OUTLIER_REJECTION} (factor={OUTLIER_FACTOR})')
print(f'  wrist anchor (5b):             {ENABLE_WRIST_ANCHOR}')
print(f'  joint-target smoothing (8a):   {ENABLE_JOINT_SMOOTHING} (fast_thresh={FAST_MOTION_THRESHOLD})')
print(f'  pose prior (6):                {ENABLE_POSE_PRIOR} (w={POSE_PRIOR_W})')
print(f'  temporal prior (8b):           {ENABLE_TEMPORAL_PRIOR} (w={TEMPORAL_W})')
print(f'  contact loss (7):              {ENABLE_CONTACT_LOSS}')

In [ ]:
import sys, os, json, pickle, types, time
import h5py
import cv2
import numpy as np
import torch
from torch.utils.data import default_collate

H5_PATH = '/data/fmalmb/DATA/tmp/Testdata.h5'
WILOR_DIR = '/home/fmalmb/CODE/WiLoR'
if WILOR_DIR not in sys.path:
    sys.path.insert(0, WILOR_DIR)

from wilor.models import load_wilor
from wilor.datasets.vitdet_dataset import ViTDetDataset
from wilor.utils import recursive_to
from wilor.utils.geometry import rot6d_to_rotmat, aa_to_rotmat

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# load_wilor() hardcodes relative './mano_data/...' paths, so it must be
# called with WILOR_DIR as the working directory.
_cwd = os.getcwd()
os.chdir(WILOR_DIR)
try:
    model, model_cfg = load_wilor(
        checkpoint_path=os.path.join(WILOR_DIR, 'pretrained_models/wilor_final.ckpt'),
        cfg_path=os.path.join(WILOR_DIR, 'pretrained_models/model_config.yaml'),
    )
finally:
    os.chdir(_cwd)

model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)

MANO_FACES = np.asarray(model.mano.faces, dtype=np.int64)
NUM_HAND_JOINTS = 15  # MANO finger joints; model.mano hand_pose has shape (B, 15, 3, 3)

# Mirroring a right-hand MANO mesh through a world plane (x -> -x) to render a left
# hand flips its chirality and therefore the winding order of MANO_FACES; reversing
# each triangle's vertex order restores outward-facing normals.
MANO_FACES_MIRRORED = MANO_FACES[:, [0, 2, 1]]

print(f'WiLoR model loaded on {DEVICE}; MANO faces: {MANO_FACES.shape}')

In [ ]:
# Load Chameleon calibration from H5 (same approach as h5_explore.ipynb).
class ChameleonCalibrationV1:
    """Stub for unpickling when calib package is not installed."""
    pass


calib_mod = types.ModuleType('calib.chameleon_calibration')
calib_mod.ChameleonCalibrationV1 = ChameleonCalibrationV1
sys.modules['calib.chameleon_calibration'] = calib_mod
sys.modules['calib'] = types.ModuleType('calib')


def build_camera_cal(intrin_vec, cam_to_world):
    cx, cy, fx, fy = intrin_vec[2], intrin_vec[3], intrin_vec[4], intrin_vec[5]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float64)
    dist = intrin_vec[6:14].astype(np.float64)
    return K, dist


with h5py.File(H5_PATH, 'r') as f:
    cal = pickle.loads(bytes(f['calibration/cal_pickle'][()]))
    camera_ids = sorted(f['rec'].keys())

CAMERA_CALIB = {}
CAMERA_WORLD_TRANSFORMS = {}
PROJECTION_MATRICES = {}
for i, cam_id in enumerate(camera_ids):
    K, dist = build_camera_cal(cal.intrinsics[i], cal.cam_to_world[i])
    w, h = cal.intrinsics[i][0], cal.intrinsics[i][1]
    CAMERA_CALIB[cam_id] = {'K': K, 'dist': dist, 'width': float(w), 'height': float(h)}
    T_cw = cal.cam_to_world[i].astype(np.float64)
    CAMERA_WORLD_TRANSFORMS[cam_id] = T_cw
    # Maps a world point to this camera's 2D pixel (dist=None convention, see
    # run_wilor_on_image below): P @ [X,Y,Z,1] ~ K @ (R_wc @ X + t_wc).
    T_wc = np.linalg.inv(T_cw)
    PROJECTION_MATRICES[cam_id] = K @ T_wc[:3, :]
    print(f"  {cam_id}: fx={K[0, 0]:.1f}, fy={K[1, 1]:.1f}, cx={K[0, 2]:.1f}, cy={K[1, 2]:.1f}")

print(f'Loaded ChameleonCalibrationV1 ({cal.n_cameras} cameras, gravity={cal.gravity_world})')

In [ ]:
with open('extracted_2d_keypoints.json') as f:
    keypoints_2d = json.load(f)

frame_keys = sorted(keypoints_2d.keys(), key=int)
NUM_FRAMES = len(frame_keys)
N_CAMS = len(camera_ids)
HAND_SIDES = ['left_hand', 'right_hand']
SIDE_INDEX = {'left_hand': 0, 'right_hand': 1}
SIDE_IS_RIGHT = {'left_hand': 0.0, 'right_hand': 1.0}
RESCALE_FACTOR = 2.0  # crop padding factor, also used by run_wilor_on_image below

print(f'{NUM_FRAMES} frames, {N_CAMS} cameras: {camera_ids}')


def hand_bbox(landmarks):
    xs = np.array([lm['x'] for lm in landmarks], dtype=np.float32)
    ys = np.array([lm['y'] for lm in landmarks], dtype=np.float32)
    return np.array([xs.min(), ys.min(), xs.max(), ys.max()], dtype=np.float32)


def hand_confidence(landmarks):
    return float(np.mean([lm['confidence'] for lm in landmarks]))


def decode_frame(h5file, cam_id, frame_key):
    jpeg = h5file[f'rec/{cam_id}/frames/{frame_key}/color'][()]
    return cv2.imdecode(np.frombuffer(jpeg, dtype=np.uint8), cv2.IMREAD_COLOR)


def hand_boxes_for(entry):
    """Return (boxes, rights, sides) for the non-empty hand detections in a cam/frame entry."""
    boxes, rights, sides = [], [], []
    for side in HAND_SIDES:
        lms = entry[side]
        if len(lms) == 21:
            boxes.append(hand_bbox(lms))
            rights.append(SIDE_IS_RIGHT[side])
            sides.append(side)
    return boxes, rights, sides


def truncation_factor(box, img_shape):
    """Fraction of WiLoR's padded square crop around `box` that falls inside the image.

    A hand near the image border has a badly-truncated crop, so its WiLoR
    prediction is down-weighted relative to cameras with a fully-visible hand.
    """
    x0, y0, x1, y1 = box
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    half = max(x1 - x0, y1 - y0) * RESCALE_FACTOR / 2.0
    px0, py0, px1, py1 = cx - half, cy - half, cx + half, cy + half
    h, w = img_shape[:2]
    ix = max(0.0, min(px1, w) - max(px0, 0.0))
    iy = max(0.0, min(py1, h) - max(py0, 0.0))
    total = (px1 - px0) * (py1 - py0)
    return float(ix * iy / total) if total > 0 else 0.0

In [ ]:
def run_wilor_on_image(img_bgr, boxes, rights, cam_id):
    """Run WiLoR on one or more hand crops from a single camera image.

    `boxes` is (N,4) xyxy pixel coordinates, `rights` is (N,) with 1.0 = right
    hand / 0.0 = left hand. Returns a list of dicts, one per crop, with:
      - 'verts_cam'  (778,3): MANO mesh vertices in this camera's 3D frame
      - 'joints_cam' (21,3):  MANO/OpenPose-order joints in this camera's 3D frame
      - 'betas'      (10,):   MANO shape parameters
      - 'is_right':  1.0 / 0.0

    Positions use the calibrated focal length / principal point (not WiLoR's
    `cam_crop_to_full`), and for left hands mirror x -> -x, so that
    `cv2.projectPoints(.., K, None)` (dist=None -- `box_center` is in raw/
    distorted pixel coordinates, so this lands at the detected pixel location
    in the raw image) reproduces the 2D detection for either hand.
    """
    dataset = ViTDetDataset(model_cfg, img_bgr, boxes, rights, rescale_factor=RESCALE_FACTOR)
    batch = default_collate([dataset[i] for i in range(len(dataset))])
    batch = recursive_to(batch, DEVICE)

    with torch.no_grad():
        out = model(batch)

    pred_cam = out['pred_cam'].clone()
    pred_cam[:, 1] = (2 * batch['right'] - 1) * pred_cam[:, 1]

    box_center = batch['box_center'].float()
    box_size = batch['box_size'].float()
    K = CAMERA_CALIB[cam_id]['K']
    fx, cx, cy = float(K[0, 0]), float(K[0, 2]), float(K[1, 2])

    bs = box_size * pred_cam[:, 0] + 1e-9
    tz = 2 * fx / bs
    tx = 2 * (box_center[:, 0] - cx) / bs + pred_cam[:, 1]
    ty = 2 * (box_center[:, 1] - cy) / bs + pred_cam[:, 2]
    cam_t = torch.stack([tx, ty, tz], dim=-1).detach().cpu().numpy()

    betas_out = out['pred_mano_params']['betas'].detach().cpu().numpy().astype(np.float32)
    global_orient_out = out['pred_mano_params']['global_orient'].detach().cpu().numpy().astype(np.float32)
    hand_pose_out = out['pred_mano_params']['hand_pose'].detach().cpu().numpy().astype(np.float32)

    results = []
    for n in range(len(dataset)):
        is_right = float(batch['right'][n].cpu().numpy())
        factor = 2 * is_right - 1
        verts = out['pred_vertices'][n].detach().cpu().numpy().astype(np.float32).copy()
        kpts = out['pred_keypoints_3d'][n].detach().cpu().numpy().astype(np.float32).copy()
        verts[:, 0] *= factor
        kpts[:, 0] *= factor
        results.append({
            'verts_cam': verts + cam_t[n],
            'joints_cam': kpts + cam_t[n],
            'betas': betas_out[n],
            # Raw MANOLayer rotation-matrix pose params, as predicted (pose2rot=False).
            # For left-hand crops these are in the network's horizontally-mirrored
            # "as if right hand" frame -- NOT yet flipped like verts/kpts above.
            'global_orient': global_orient_out[n, 0],  # (3,3)
            'hand_pose': hand_pose_out[n],  # (15,3,3)
            'is_right': is_right,
        })
    return results

In [ ]:
OUT_RAW_NPZ = 'wilor_hand_meshes_raw_v2.npz'

joints_cam_all = np.zeros((NUM_FRAMES, N_CAMS, 2, 21, 3), dtype=np.float32)
betas_all = np.zeros((NUM_FRAMES, N_CAMS, 2, 10), dtype=np.float32)
global_orient_all = np.zeros((NUM_FRAMES, N_CAMS, 2, 3, 3), dtype=np.float32)
hand_pose_all = np.zeros((NUM_FRAMES, N_CAMS, 2, 15, 3, 3), dtype=np.float32)
valid_all = np.zeros((NUM_FRAMES, N_CAMS, 2), dtype=bool)
weight_all = np.zeros((NUM_FRAMES, N_CAMS, 2), dtype=np.float32)

t0 = time.time()
with h5py.File(H5_PATH, 'r') as f5:
    for fi, frame_key in enumerate(frame_keys):
        for ci, cam_id in enumerate(camera_ids):
            entry = keypoints_2d[frame_key][cam_id]
            boxes, rights, sides = hand_boxes_for(entry)
            if not boxes:
                continue
            img = decode_frame(f5, cam_id, frame_key)
            results = run_wilor_on_image(img, np.stack(boxes), np.array(rights, dtype=np.float32), cam_id)
            for res, side, box in zip(results, sides, boxes):
                si = SIDE_INDEX[side]
                joints_cam_all[fi, ci, si] = res['joints_cam']
                betas_all[fi, ci, si] = res['betas']
                global_orient_all[fi, ci, si] = res['global_orient']
                hand_pose_all[fi, ci, si] = res['hand_pose']
                valid_all[fi, ci, si] = True
                weight_all[fi, ci, si] = hand_confidence(entry[side]) * truncation_factor(box, img.shape)
        if (fi + 1) % 16 == 0 or (fi + 1) == NUM_FRAMES:
            print(f'  {fi + 1}/{NUM_FRAMES} frames ({time.time() - t0:.1f}s)')

np.savez_compressed(
    OUT_RAW_NPZ,
    joints_cam=joints_cam_all,
    betas=betas_all,
    global_orient=global_orient_all,
    hand_pose=hand_pose_all,
    valid=valid_all,
    weight=weight_all,
    mano_faces=MANO_FACES,
    frame_keys=np.array(frame_keys),
    camera_ids=np.array(camera_ids),
)
print(f'Saved {OUT_RAW_NPZ}')
print('Coverage (valid detections) per camera/side:')
for ci, cam_id in enumerate(camera_ids):
    print(f"  {cam_id}: left={valid_all[:, ci, 0].sum():3d}/{NUM_FRAMES}  right={valid_all[:, ci, 1].sum():3d}/{NUM_FRAMES}")

# Spot-check vs v1's raw output: frame 0, cam00, right hand wrist (joint 0).
v1_raw = np.load('wilor_hand_meshes_raw.npz', allow_pickle=True)
print('spot-check joint 0 (wrist), frame 0, cam00, right hand:')
print('  v2 joints_cam[0,0,1,0] =', joints_cam_all[0, 0, 1, 0])
print('  v1 wrist_cam[0,0,1]    =', v1_raw['wrist_cam'][0, 0, 1])

In [ ]:
# === Physical-hand identity clustering (step 1b) =====================================
# MediaPipe's per-camera left/right label for the SAME physical hand is often
# inconsistent -- both across cameras at a given frame and across time for a given
# camera (87% of frames have at least one camera's label disagreeing with the
# consensus for that hand). WiLoR's is_right/mirroring is taken directly from that
# label, so the inconsistency propagates into joints_cam/global_orient/hand_pose too.
#
# This cell replaces the raw `si` axis (MediaPipe's per-camera guess) with a `pid`
# axis (physical-hand identity, 0 or 1) derived from 3D proximity of each camera's
# wrist (joint 0) detection, with one constraint: a single camera's own
# left_hand/right_hand detections (if both valid in the same frame) must be different
# physical hands -- MediaPipe found two distinct hands in that view even if its labels
# for them disagree with other cameras.
#
# For (frame, camera) entries whose raw label disagrees with the pid consensus, the
# already-extracted joints_cam/global_orient/hand_pose are re-mirrored (the same
# x -> -x / M @ R @ M operation that turns WiLoR's "as if right hand" network output
# into a left-hand reconstruction) so they describe the consensus physical hand under
# the same convention as every other camera's contribution to that pid.
#
# Everything downstream (06+) keeps using `si`/HAND_SIDES/SIDE_INDEX as before -- they
# are redefined here from the pid-consensus vote, and valid_all/weight_all/
# joints_cam_all/betas_all/global_orient_all/hand_pose_all are replaced by their
# pid-indexed counterparts, so the rest of the pipeline is unchanged.

MIRROR_X = np.diag([-1.0, 1.0, 1.0])


def _world_wrist_positions(joints_cam_all, valid_all):
    n_frames, n_cams, n_sides = valid_all.shape
    wpos = np.full((n_frames, n_cams, n_sides, 3), np.nan, dtype=np.float64)
    for ci, cam_id in enumerate(camera_ids):
        T = CAMERA_WORLD_TRANSFORMS[cam_id]
        R, t = T[:3, :3], T[:3, 3]
        for fi in range(n_frames):
            for si in range(n_sides):
                if valid_all[fi, ci, si]:
                    wpos[fi, ci, si] = R @ joints_cam_all[fi, ci, si, 0] + t
    return wpos


def _cluster_frame(points, weights, cannot_link, prev_centroids):
    """Constrained 2-means: returns (labels in {0,1}, centroids (2,3))."""
    n = len(points)
    if n == 1:
        if prev_centroids is not None:
            d0 = np.linalg.norm(points[0] - prev_centroids[0])
            d1 = np.linalg.norm(points[0] - prev_centroids[1])
            label = 0 if d0 <= d1 else 1
            centroids = prev_centroids.copy()
        else:
            label = 0
            centroids = np.array([points[0], points[0]])
        centroids[label] = points[0]
        return np.array([label]), centroids

    if prev_centroids is not None:
        centroids = prev_centroids.copy()
    elif cannot_link:
        i, j = cannot_link[0]
        centroids = np.array([points[i], points[j]])
    else:
        d = np.linalg.norm(points[:, None, :] - points[None, :, :], axis=-1)
        i, j = np.unravel_index(np.argmax(d), d.shape)
        centroids = np.array([points[i], points[j]])

    labels = np.zeros(n, dtype=int)
    for _ in range(5):
        d0 = np.linalg.norm(points - centroids[0], axis=1)
        d1 = np.linalg.norm(points - centroids[1], axis=1)
        labels = (d1 < d0).astype(int)

        for i, j in cannot_link:
            if labels[i] == labels[j]:
                margin_i = abs(d0[i] - d1[i])
                margin_j = abs(d0[j] - d1[j])
                if margin_i <= margin_j:
                    labels[i] = 1 - labels[i]
                else:
                    labels[j] = 1 - labels[j]

        new_centroids = centroids.copy()
        for k in (0, 1):
            mask = labels == k
            if mask.any():
                new_centroids[k] = np.average(points[mask], axis=0, weights=weights[mask])
        if np.allclose(new_centroids, centroids):
            centroids = new_centroids
            break
        centroids = new_centroids

    return labels, centroids


def cluster_physical_hands(joints_cam_all, valid_all, weight_all):
    n_frames, n_cams, n_sides = valid_all.shape
    wpos = _world_wrist_positions(joints_cam_all, valid_all)

    pid_all = np.full((n_frames, n_cams, n_sides), -1, dtype=np.int8)
    centroids_all = np.full((n_frames, 2, 3), np.nan, dtype=np.float64)

    prev_centroids = None
    for fi in range(n_frames):
        idx = [(ci, si) for ci in range(n_cams) for si in range(n_sides) if valid_all[fi, ci, si]]
        if not idx:
            prev_centroids = None
            continue
        points = np.array([wpos[fi, ci, si] for ci, si in idx])
        weights = np.array([max(weight_all[fi, ci, si], 1e-6) for ci, si in idx])

        index_of = {pair: k for k, pair in enumerate(idx)}
        cannot_link = []
        for ci in range(n_cams):
            if (ci, 0) in index_of and (ci, 1) in index_of:
                cannot_link.append((index_of[(ci, 0)], index_of[(ci, 1)]))

        labels, centroids = _cluster_frame(points, weights, cannot_link, prev_centroids)

        if prev_centroids is not None and not np.isnan(prev_centroids).any():
            d_same = np.linalg.norm(centroids - prev_centroids, axis=1).sum()
            d_swap = np.linalg.norm(centroids[::-1] - prev_centroids, axis=1).sum()
            if d_swap < d_same:
                labels = 1 - labels
                centroids = centroids[::-1]

        for (ci, si), lab in zip(idx, labels):
            pid_all[fi, ci, si] = lab
        centroids_all[fi] = centroids
        prev_centroids = centroids

    # global left/right label per pid via weighted majority vote over raw si
    votes = np.zeros((2, n_sides), dtype=np.float64)
    for fi in range(n_frames):
        for ci in range(n_cams):
            for si in range(n_sides):
                pid = pid_all[fi, ci, si]
                if pid >= 0:
                    votes[pid, si] += weight_all[fi, ci, si]

    pid_to_side = {}
    if votes[0, 1] + votes[1, 0] >= votes[0, 0] + votes[1, 1]:
        pid_to_side[0], pid_to_side[1] = HAND_SIDES_RAW[1], HAND_SIDES_RAW[0]
    else:
        pid_to_side[0], pid_to_side[1] = HAND_SIDES_RAW[0], HAND_SIDES_RAW[1]

    return pid_all, pid_to_side, centroids_all


HAND_SIDES_RAW = list(HAND_SIDES)  # ['left_hand', 'right_hand'] -- MediaPipe's raw si order
pid_all, pid_to_side, centroids_all = cluster_physical_hands(joints_cam_all, valid_all, weight_all)
print('pid_to_side:', pid_to_side)

# --- remap every per-(frame, camera, si) array onto the pid axis, re-mirroring
# joints_cam/global_orient/hand_pose for entries whose raw label disagrees with the
# pid consensus ---
joints_cam_pid = np.zeros_like(joints_cam_all)
betas_pid = np.zeros_like(betas_all)
global_orient_pid = np.zeros_like(global_orient_all)
hand_pose_pid = np.zeros_like(hand_pose_all)
valid_pid = np.zeros_like(valid_all)
weight_pid = np.zeros_like(weight_all)

n_remirrored = 0
for fi in range(NUM_FRAMES):
    for ci in range(N_CAMS):
        for si in range(2):
            pid = pid_all[fi, ci, si]
            if pid < 0:
                continue
            jc, go, hp = joints_cam_all[fi, ci, si], global_orient_all[fi, ci, si], hand_pose_all[fi, ci, si]
            if HAND_SIDES_RAW[si] != pid_to_side[pid]:
                jc = jc * np.array([-1.0, 1.0, 1.0], dtype=jc.dtype)
                go = (MIRROR_X @ go.astype(np.float64) @ MIRROR_X).astype(go.dtype)
                hp = np.einsum('ij,kjl,lm->kim', MIRROR_X, hp.astype(np.float64), MIRROR_X).astype(hp.dtype)
                n_remirrored += 1
            joints_cam_pid[fi, ci, pid] = jc
            betas_pid[fi, ci, pid] = betas_all[fi, ci, si]
            global_orient_pid[fi, ci, pid] = go
            hand_pose_pid[fi, ci, pid] = hp
            valid_pid[fi, ci, pid] = True
            weight_pid[fi, ci, pid] = weight_all[fi, ci, si]

n_relabeled = int((valid_pid.sum()))
print(f'remapped {n_relabeled} detections onto the pid axis '
      f'({n_remirrored} re-mirrored where raw MediaPipe label != pid consensus)')

joints_cam_all, betas_all, global_orient_all, hand_pose_all, valid_all, weight_all = (
    joints_cam_pid, betas_pid, global_orient_pid, hand_pose_pid, valid_pid, weight_pid)

# Redefine HAND_SIDES/SIDE_INDEX from the pid-consensus labels so downstream cells
# (06+) operate on `pid` through the same si-shaped interface as before.
HAND_SIDES = [pid_to_side[0], pid_to_side[1]]
SIDE_INDEX = {side: pid for pid, side in pid_to_side.items()}

print('pid coverage per camera:')
for ci, cam_id in enumerate(camera_ids):
    print(f"  {cam_id}: {pid_to_side[0]}={int(valid_all[:, ci, 0].sum()):3d}/{NUM_FRAMES}  "
          f"{pid_to_side[1]}={int(valid_all[:, ci, 1].sum()):3d}/{NUM_FRAMES}")

In [ ]:
# === Camera calibration quality weights ==============================================
# This rig's calibration (CAMERA_CALIB/CAMERA_WORLD_TRANSFORMS, loaded in calib02 from
# the H5's calibration/cal_pickle) has substantial *systematic, per-camera* reprojection
# error. Verified by reprojecting final_reconstructed_3d_keypoints.json's triangulated
# body joints (independent of WiLoR/hands entirely) back into each camera and comparing
# to that camera's own raw MediaPipe 2D detection (undistorted): the error is
# frame-independent (same pattern at frames 0/30/60/90/120) and camera-specific, e.g.
# cam04 ~20px vs. cam03 ~300px on a 3360x1890 image -- a few cm to ~15cm of per-camera
# bias, baked into the provided calibration itself (not introduced by this pipeline).
#
# Each hand's fused position/orientation below combines whichever cameras currently see
# it -- a set that changes every frame -- so this per-camera bias is a direct cause of
# "wrong orientation, jumping" symptoms: cameras with large extrinsics error get equal
# say with accurate ones, and the mix of contributing cameras changes frame to frame.
# CALIB_WEIGHT downweights the less-accurate cameras in Simple AVG's rotation average.
# The wrist *position* itself is anchored to the body-pose triangulation
# (final_reconstructed_3d_keypoints.json, same as the wrist anchor in triang09), which
# already combines all 7 cameras' body keypoints per frame and is far more stable than
# any single-hand-camera world-frame estimate.

def undistort_pixel(uv, K, dist):
    pt = np.asarray(uv, dtype=np.float64).reshape(1, 1, 2)
    return cv2.undistortPoints(pt, K, dist, P=K).reshape(2)


with open('final_reconstructed_3d_keypoints.json') as f:
    body_3d = json.load(f)
BODY_WRIST_JOINT = {SIDE_INDEX['left_hand']: '15', SIDE_INDEX['right_hand']: '16'}

_CALIBQC_BODY_IDS = [str(i) for i in range(9)]  # head/shoulders: dense, high-confidence
_CALIBQC_FRAME_STRIDE = 4

cam_errs = {cam_id: [] for cam_id in camera_ids}
for fi in range(0, NUM_FRAMES, _CALIBQC_FRAME_STRIDE):
    frame_key = frame_keys[fi]
    frame_body3d = body_3d.get(frame_key, {})
    for body_id in _CALIBQC_BODY_IDS:
        ref = frame_body3d.get(body_id)
        if ref is None or ref['combined_confidence'] < 0.9:
            continue
        X = np.array([ref['X'], ref['Y'], ref['Z']])
        for cam_id in camera_ids:
            lm = next((p for p in keypoints_2d[frame_key][cam_id]['body'] if p['id'] == int(body_id)), None)
            if lm is None or lm['confidence'] < 0.8:
                continue
            K, dist = CAMERA_CALIB[cam_id]['K'], CAMERA_CALIB[cam_id]['dist']
            px = undistort_pixel([lm['x'], lm['y']], K, dist)
            proj = PROJECTION_MATRICES[cam_id] @ np.append(X, 1.0)
            cam_errs[cam_id].append(np.linalg.norm(proj[:2] / proj[2] - px))

CALIB_WEIGHT = {}
print('Per-camera calibration reprojection error vs. final_reconstructed_3d_keypoints.json:')
for cam_id in camera_ids:
    mean_err = float(np.mean(cam_errs[cam_id]))
    CALIB_WEIGHT[cam_id] = 1.0 / mean_err
    print(f'  {cam_id}: mean reproj err = {mean_err:6.1f}px  (n={len(cam_errs[cam_id])})')

_w_mean = np.mean(list(CALIB_WEIGHT.values()))
for cam_id in camera_ids:
    CALIB_WEIGHT[cam_id] /= _w_mean
print('  calib weights (normalized, mean=1):', {k: round(v, 3) for k, v in CALIB_WEIGHT.items()})

In [ ]:
# === Simple AVG Debug ================================================================
# A deliberately "dumb" baseline, independent of steps 4-8 below, for comparing
# against the fitted pipeline:
#   - betas: ONE (10,) vector per hand for the whole sequence -- the mean of every
#     valid (frame, camera) betas prediction, no per-frame or temporal fusion.
#   - per frame: cameras with weight >= SIMPLE_AVG_CONF_THRESHOLD are selected
#     (falling back to all valid cameras for that frame if none clear the bar).
#     global_orient and hand_pose are each the selected cameras' raw rotation
#     matrices, averaged with weight = mediapipe_confidence * CALIB_WEIGHT[cam] (see
#     calibqc056) and re-projected to the nearest rotation matrix (orthogonal
#     Procrustes) -- no outlier rejection, no temporal smoothing, no joint-position
#     fitting.
#   - wrist position: taken directly from the body-pose triangulation
#     (final_reconstructed_3d_keypoints.json, same anchor as triang09's joint 0).
#     Mapping each camera's own wrist joint (joints_cam[...,0]) to world via that
#     camera's extrinsics and averaging was tried first, but this calibration's
#     per-camera bias (calibqc056) makes that average land tens of cm off and jump
#     around as the contributing camera set changes frame to frame; the body-pose
#     triangulation already fuses all 7 cameras per frame and is far more stable.
#
# WiLoR predicts global_orient/hand_pose for left-hand crops in a horizontally-
# mirrored ("as if right hand") frame. hand_pose is intrinsic/joint-local, so it can
# be averaged as-is; global_orient needs the camera's world rotation conjugated by
# MIRROR_X = diag(-1,1,1) before averaging, and the rendered mesh is mirrored back --
# same trick as reflect() in the MANO-fit cell below.

SIMPLE_AVG_CONF_THRESHOLD = 0.3
MIRROR_X = np.diag([-1.0, 1.0, 1.0])
IDENTITY_3 = np.eye(3)

def orthogonalize(mats):
    # Project averaged 3x3 matrices to the nearest proper rotation matrices.
    shape = mats.shape
    mats = mats.reshape(-1, 3, 3)
    U, _, Vt = np.linalg.svd(mats)
    det = np.linalg.det(U @ Vt)
    U[det < 0, :, -1] *= -1
    return (U @ Vt).reshape(shape)

# --- single beta per hand, averaged over every valid (frame, camera) ---
simple_beta = np.zeros((2, 10), dtype=np.float64)
for si in range(2):
    mask = valid_all[:, :, si] & (weight_all[:, :, si] >= SIMPLE_AVG_CONF_THRESHOLD)
    if not mask.any():
        mask = valid_all[:, :, si]
    simple_beta[si] = betas_all[:, :, si][mask].astype(np.float64).mean(axis=0)

# --- per-frame rotation + wrist averaging ---
simple_joints = np.zeros((NUM_FRAMES, 2, 21, 3), dtype=np.float64)
simple_verts = np.zeros((NUM_FRAMES, 2, 778, 3), dtype=np.float64)
simple_valid = np.zeros((NUM_FRAMES, 2), dtype=bool)
n_fallback = {0: 0, 1: 0}

n_no_anchor = {0: 0, 1: 0}
for fi in range(NUM_FRAMES):
    frame_key = frame_keys[fi]
    for si in range(2):
        side = HAND_SIDES[si]
        cams = np.where(valid_all[fi, :, si] & (weight_all[fi, :, si] >= SIMPLE_AVG_CONF_THRESHOLD))[0]
        if len(cams) == 0:
            cams = np.where(valid_all[fi, :, si])[0]
            n_fallback[si] += 1
        if len(cams) == 0:
            continue

        M = MIRROR_X if side == 'left_hand' else IDENTITY_3
        global_orient_terms, hand_pose_terms, cam_weights = [], [], []
        for ci in cams:
            cam_id = camera_ids[ci]
            T_c2w = CAMERA_WORLD_TRANSFORMS[cam_id]
            R_c2w = T_c2w[:3, :3]
            G_raw = global_orient_all[fi, ci, si].astype(np.float64)
            global_orient_terms.append(M @ R_c2w @ M @ G_raw)
            hand_pose_terms.append(hand_pose_all[fi, ci, si].astype(np.float64))
            cam_weights.append(weight_all[fi, ci, si] * CALIB_WEIGHT[cam_id])

        cam_weights = np.array(cam_weights, dtype=np.float64)
        cam_weights /= cam_weights.sum()
        global_orient_avg = orthogonalize(np.tensordot(cam_weights, np.array(global_orient_terms), axes=(0, 0)))
        hand_pose_avg = orthogonalize(np.tensordot(cam_weights, np.array(hand_pose_terms), axes=(0, 0)))

        ref = body_3d.get(frame_key, {}).get(BODY_WRIST_JOINT[si])
        if ref is not None:
            wrist_world_avg = np.array([ref['X'], ref['Y'], ref['Z']])
        else:
            n_no_anchor[si] += 1
            wrist_world_terms = []
            for ci in cams:
                cam_id = camera_ids[ci]
                T_c2w = CAMERA_WORLD_TRANSFORMS[cam_id]
                wrist_cam = joints_cam_all[fi, ci, si, 0].astype(np.float64)
                wrist_world_terms.append(T_c2w[:3, :3] @ wrist_cam + T_c2w[:3, 3])
            wrist_world_avg = np.mean(wrist_world_terms, axis=0)

        betas_t = torch.tensor(simple_beta[si], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        global_orient_t = torch.tensor(global_orient_avg, dtype=torch.float32, device=DEVICE).view(1, 1, 3, 3)
        hand_pose_t = torch.tensor(hand_pose_avg, dtype=torch.float32, device=DEVICE).view(1, NUM_HAND_JOINTS, 3, 3)
        with torch.no_grad():
            out = model.mano(betas=betas_t, global_orient=global_orient_t, hand_pose=hand_pose_t, pose2rot=False)
        pred_joints = out.joints[0].cpu().numpy().astype(np.float64)
        pred_verts = out.vertices[0].cpu().numpy().astype(np.float64)

        if side == 'left_hand':
            pred_joints = pred_joints.copy()
            pred_verts = pred_verts.copy()
            pred_joints[:, 0] *= -1
            pred_verts[:, 0] *= -1

        transl = wrist_world_avg - pred_joints[0]
        simple_joints[fi, si] = pred_joints + transl
        simple_verts[fi, si] = pred_verts + transl
        simple_valid[fi, si] = True

print('Simple AVG Debug:')
print(f'  conf threshold: {SIMPLE_AVG_CONF_THRESHOLD}')
print(f'  left valid: {simple_valid[:, 0].sum()}/{NUM_FRAMES}  right valid: {simple_valid[:, 1].sum()}/{NUM_FRAMES}')
print(f'  frames falling back to all valid cams (none met threshold): '
      f'left={n_fallback[0]}, right={n_fallback[1]}')
print(f'  frames with no body-pose wrist anchor (fell back to per-camera depth avg): '
      f'left={n_no_anchor[0]}, right={n_no_anchor[1]}')

li, ri = SIDE_INDEX['left_hand'], SIDE_INDEX['right_hand']
np.savez_compressed(
    'simple_avg_debug_v2.npz',
    left_verts=simple_verts[:, li].astype(np.float32),
    right_verts=simple_verts[:, ri].astype(np.float32),
    left_valid=simple_valid[:, li],
    right_valid=simple_valid[:, ri],
    mano_faces=MANO_FACES,
    mano_faces_mirrored=MANO_FACES_MIRRORED,
    frame_keys=np.array(frame_keys),
)
print('Saved simple_avg_debug_v2.npz')

In [ ]:
import viser
from viser import transforms as tf

with open('final_reconstructed_3d_keypoints.json') as f:
    body_3d = json.load(f)

_hand_data = np.load('simple_avg_debug_v2.npz', allow_pickle=True)
left_verts = _hand_data['left_verts']
right_verts = _hand_data['right_verts']
left_valid = _hand_data['left_valid']
right_valid = _hand_data['right_valid']

# Calibration world is Y-up; Viser uses Z-up. Flip world Y so the figure is upright.
R_WORLD_TO_VISER = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, -1.0, 0.0],
], dtype=np.float64)
FRUSTUM_SCALE_BASE = 0.10
FRUSTUM_SCALE_PER_M = 0.03

POSE_CONNECTIONS = [
    (11, 12), (11, 23), (12, 24), (23, 24),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
    (0, 1), (1, 2), (2, 3), (0, 4), (4, 5), (5, 6), (3, 7), (6, 8),
    (9, 10),
]
POSE_HAND_JOINTS = {17, 18, 19, 20, 21, 22}


def skeleton_origin(joints_dict):
    """Pelvis midpoint, else mean of all body joints."""
    hip_pts = []
    for hip_id in (23, 24):
        key = str(hip_id)
        if key in joints_dict:
            hip_pts.append([joints_dict[key]['X'], joints_dict[key]['Y'], joints_dict[key]['Z']])
    if len(hip_pts) == 2:
        return np.mean(hip_pts, axis=0)
    body_pts = [[v['X'], v['Y'], v['Z']] for k, v in joints_dict.items() if int(k) < 100]
    if body_pts:
        return np.mean(body_pts, axis=0)
    return np.zeros(3, dtype=np.float64)


def world_to_viser_point(p_world, origin_world):
    return R_WORLD_TO_VISER @ (np.asarray(p_world, dtype=np.float64) - origin_world)


def world_to_viser_pose(T_cw, origin_world):
    T = np.asarray(T_cw, dtype=np.float64).copy()
    T[:3, 3] = R_WORLD_TO_VISER @ (T[:3, 3] - origin_world)
    T[:3, :3] = R_WORLD_TO_VISER @ T[:3, :3]
    wxyz = tf.SO3.from_matrix(T[:3, :3]).wxyz
    return wxyz.astype(np.float64), T[:3, 3].astype(np.float64)


def body_joint_coords(joints_dict, origin_world):
    out = {}
    for j_id, v in joints_dict.items():
        jid = int(j_id)
        if jid >= 100 or jid in POSE_HAND_JOINTS:
            continue
        out[jid] = world_to_viser_point([v['X'], v['Y'], v['Z']], origin_world).astype(np.float32)
    return out


def add_bones(server, name, joint_coords, connections, color):
    segs = []
    for a, b in connections:
        if a in joint_coords and b in joint_coords:
            segs.append([joint_coords[a], joint_coords[b]])
    if segs:
        server.scene.add_line_segments(
            name=name,
            points=np.array(segs, dtype=np.float32),
            colors=np.array(color, dtype=np.uint8),
            line_width=2.0,
        )


def apply_fov(server, fov_deg):
    fov_rad = float(np.deg2rad(fov_deg))
    server.initial_camera.fov = fov_rad
    for client in server.get_clients().values():
        client.camera.fov = fov_rad


def verts_to_viser(verts_world, origin_world):
    return (R_WORLD_TO_VISER @ (verts_world - origin_world).T).T


ORIGIN_WORLD = skeleton_origin(body_3d[frame_keys[0]])

HAND_MESH_SPECS = [
    ('left_hand', left_verts, left_valid, MANO_FACES_MIRRORED, (180, 220, 255), '/hands/left_simpleavg'),
    ('right_hand', right_verts, right_valid, MANO_FACES, (255, 200, 170), '/hands/right_simpleavg'),
]


def setup_cameras(server):
    for cam_id, T_cw in CAMERA_WORLD_TRANSFORMS.items():
        calib = CAMERA_CALIB[cam_id]
        K = calib['K']
        fov = 2.0 * np.arctan(calib['height'] / (2.0 * K[1, 1]))
        aspect = float(calib['width'] / calib['height'])
        wxyz, position = world_to_viser_pose(T_cw, ORIGIN_WORLD)
        dist = float(np.linalg.norm(position))
        frustum_scale = float(np.clip(FRUSTUM_SCALE_BASE + FRUSTUM_SCALE_PER_M * dist, 0.08, 0.16))
        server.scene.add_camera_frustum(
            name=f'/cameras/{cam_id}',
            fov=float(fov),
            aspect=aspect,
            scale=frustum_scale,
            line_width=0.5,
            color=(80, 160, 255),
            wxyz=wxyz,
            position=position,
        )
        server.scene.add_frame(
            name=f'/cameras/{cam_id}/axis',
            axes_length=0.08,
            axes_radius=0.004,
            wxyz=wxyz,
            position=position,
        )


def main():
    server = viser.ViserServer()
    running = True
    mesh_handles = {}
    try:
        server.scene.set_up_direction('+z')
        server.scene.configure_environment_map(hdri=None, background=False, environment_intensity=0.6)
        server.gui.configure_theme(dark_mode=True)
        server.scene.set_background_image(np.zeros((8, 8, 3), dtype=np.uint8))

        setup_cameras(server)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print('Viser (Simple AVG Debug): open the URL shown above')
        print(f'Origin offset (calibration world): {ORIGIN_WORLD}')

        play_button = server.gui.add_button('Play / Pause')
        stop_button = server.gui.add_button('Stop viewer')
        speed_slider = server.gui.add_slider('Playback Speed (FPS)', min=1, max=60, step=1, initial_value=15)
        frame_slider = server.gui.add_slider('Timeline Frame', min=0, max=NUM_FRAMES - 1, step=1, initial_value=0)
        fov_slider = server.gui.add_slider('Camera FOV (deg)', min=20.0, max=120.0, step=1.0, initial_value=75.0)
        apply_fov(server, fov_slider.value)
        is_playing = False

        @play_button.on_click
        def _(_) -> None:
            nonlocal is_playing
            is_playing = not is_playing

        @stop_button.on_click
        def _stop(_) -> None:
            nonlocal running
            running = False

        @fov_slider.on_update
        def _fov(_) -> None:
            apply_fov(server, fov_slider.value)

        current_idx = 0
        while running:
            if not is_playing:
                current_idx = frame_slider.value
            f_key = frame_keys[current_idx]

            joint_coords = body_joint_coords(body_3d[f_key], ORIGIN_WORLD)
            if joint_coords:
                ids = sorted(joint_coords)
                pts = np.stack([joint_coords[i] for i in ids]).astype(np.float32)
                server.scene.add_point_cloud(
                    name='/skeleton/joints',
                    points=pts,
                    colors=np.full((len(ids), 3), 220, dtype=np.uint8),
                    point_size=0.005,
                )
                add_bones(server, '/skeleton/body', joint_coords, POSE_CONNECTIONS, (0, 255, 120))

            for side, verts, valid, faces, color, name in HAND_MESH_SPECS:
                if valid[current_idx]:
                    v = verts_to_viser(verts[current_idx], ORIGIN_WORLD).astype(np.float32)
                    mesh_handles[name] = server.scene.add_mesh_simple(
                        name=name, vertices=v, faces=faces, color=color, opacity=0.9,
                    )
                elif name in mesh_handles:
                    mesh_handles[name].visible = False

            if is_playing:
                current_idx = (current_idx + 1) % NUM_FRAMES
                frame_slider.value = current_idx
                time.sleep(1.0 / speed_slider.value)
            else:
                time.sleep(0.05)
    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()

In [ ]:
beta_mean = np.zeros((NUM_FRAMES, 2, 10), dtype=np.float64)
for fi in range(NUM_FRAMES):
    for si in range(2):
        cams = np.where(valid_all[fi, :, si])[0]
        if ENABLE_CONFIDENCE_WEIGHTING:
            w = weight_all[fi, cams, si].astype(np.float64)
        else:
            w = np.ones(len(cams), dtype=np.float64)
        beta_mean[fi, si] = np.tensordot(w / w.sum(), betas_all[fi, cams, si], axes=(0, 0))

beta_fused = np.zeros_like(beta_mean)
beta_fused[0] = beta_mean[0]
for fi in range(1, NUM_FRAMES):
    beta_fused[fi] = BETA_EMA_ALPHA * beta_mean[fi] + (1 - BETA_EMA_ALPHA) * beta_fused[fi - 1]

print('beta_fused shape:', beta_fused.shape)
print('frames with zero valid cams (left, right):',
      (~valid_all[:, :, 0].any(axis=1)).sum(), (~valid_all[:, :, 1].any(axis=1)).sum())
print('beta_fused frame-to-frame std (left, right):',
      np.round(beta_fused[:, 0].std(axis=0).mean(), 4), np.round(beta_fused[:, 1].std(axis=0).mean(), 4))

In [ ]:
def triangulate_point_dlt_weighted(Ps, pixels, weights):
    """Weighted multi-view DLT triangulation (same SVD approach as h5_explore.ipynb,
    with each pair of equations scaled by sqrt(weight)). Used for joint 0 (wrist)
    only -- see step 5 in the intro for why joints 1-20 are fused differently."""
    A = []
    for P, (x, y), wgt in zip(Ps, pixels, weights):
        sw = np.sqrt(wgt)
        A.append(sw * (x * P[2, :] - P[0, :]))
        A.append(sw * (y * P[2, :] - P[1, :]))
    A = np.array(A)
    _, _, Vh = np.linalg.svd(A)
    X_h = Vh[-1]
    return X_h[:3] / X_h[3]


def reproj_error_px(X, P, pixel):
    proj = P @ np.append(X, 1.0)
    return np.linalg.norm(proj[:2] / proj[2] - np.asarray(pixel))


def undistort_pixel(uv, K, dist):
    pt = np.asarray(uv, dtype=np.float64).reshape(1, 1, 2)
    return cv2.undistortPoints(pt, K, dist, P=K).reshape(2)


with open('final_reconstructed_3d_keypoints.json') as f:
    body_3d = json.load(f)
BODY_WRIST_JOINT = {SIDE_INDEX['left_hand']: '15', SIDE_INDEX['right_hand']: '16'}

joints3d_fused = np.zeros((NUM_FRAMES, 2, 21, 3), dtype=np.float64)
joints3d_valid = np.zeros((NUM_FRAMES, 2), dtype=bool)

reproj_errors = []
n_rejected = n_total = 0
anchor_offsets = []

for fi in range(NUM_FRAMES):
    for si in range(2):
        cams = np.where(valid_all[fi, :, si])[0]
        if len(cams) == 0:
            continue
        joints3d_valid[fi, si] = True

        # --- joint 0 (wrist): per-camera weighted DLT triangulation ---
        # This calibration has a baseline reprojection residual of several tens to a
        # few hundred pixels even for "correct" triangulations (verified against
        # h5_explore's own body-joint triangulations), so outlier rejection is
        # relative to the median rather than an absolute pixel threshold: only
        # cameras that disagree much more than their peers are dropped.
        Ps, pixels, weights = [], [], []
        for ci in cams:
            cam_id = camera_ids[ci]
            K, dist = CAMERA_CALIB[cam_id]['K'], CAMERA_CALIB[cam_id]['dist']
            pt_cam = joints_cam_all[fi, ci, si, 0].reshape(1, 1, 3).astype(np.float64)
            uv, _ = cv2.projectPoints(pt_cam, np.zeros(3), np.zeros(3), K, None)
            Ps.append(PROJECTION_MATRICES[cam_id])
            pixels.append(undistort_pixel(uv.reshape(2), K, dist))
            weights.append(weight_all[fi, ci, si] if ENABLE_CONFIDENCE_WEIGHTING else 1.0)

        X_dlt = triangulate_point_dlt_weighted(Ps, pixels, weights)
        errs = np.array([reproj_error_px(X_dlt, P, px) for P, px in zip(Ps, pixels)])
        keep = errs <= OUTLIER_FACTOR * np.median(errs)
        if ENABLE_TRIANG_OUTLIER_REJECTION and not keep.all():
            Ps_k, pixels_k, weights_k = (
                [v for v, k in zip(Ps, keep) if k],
                [v for v, k in zip(pixels, keep) if k],
                [v for v, k in zip(weights, keep) if k],
            )
            X_dlt = triangulate_point_dlt_weighted(Ps_k, pixels_k, weights_k)
            errs = np.array([reproj_error_px(X_dlt, P, px) for P, px in zip(Ps_k, pixels_k)])
            n_rejected += int((~keep).sum())
        n_total += len(cams)
        reproj_errors.extend(errs.tolist())

        ref = body_3d.get(frame_keys[fi], {}).get(BODY_WRIST_JOINT[si])
        if ENABLE_WRIST_ANCHOR and ref is not None:
            X_wrist = np.array([ref['X'], ref['Y'], ref['Z']])
            anchor_offsets.append(np.linalg.norm(X_wrist - X_dlt) * 1000)
        else:
            X_wrist = X_dlt
        joints3d_fused[fi, si, 0] = X_wrist

        # --- joints 1-20: take relative offsets from Simple AVG's rotation-averaged
        # MANO pose (simpleavg06: hand_pose/global_orient averaged in SO(3) across
        # cameras via orthogonal Procrustes), re-based at the triangulated wrist.
        # Per-camera *position* offsets (the previous approach here) blend
        # mutually-incompatible finger configurations into a geometrically
        # meaningless average whenever cameras disagree about articulation -- and
        # deviation-from-median outlier rejection can't catch a roughly-even
        # bimodal split (e.g. 4 cams vs. 3 cams), which happens often. Averaging in
        # rotation space instead always yields a kinematically valid hand shape.
        for j in range(1, 21):
            joints3d_fused[fi, si, j] = X_wrist + (simple_joints[fi, si, j] - simple_joints[fi, si, 0])

print(f'joints3d_fused shape: {joints3d_fused.shape}')
print(f'joint-0 (wrist) DLT mean reprojection error: {np.mean(reproj_errors):.2f}px (n={len(reproj_errors)}), '
      f'rejected {n_rejected}/{n_total} ({100 * n_rejected / n_total:.2f}%)')

print(f'left valid: {joints3d_valid[:, 0].sum()}/{NUM_FRAMES}  right valid: {joints3d_valid[:, 1].sum()}/{NUM_FRAMES}')

if anchor_offsets:
    anchor_offsets = np.array(anchor_offsets)
    print(f'wrist anchor: mean correction vs. DLT {anchor_offsets.mean():.1f}mm, max {anchor_offsets.max():.1f}mm '
          f'(n={len(anchor_offsets)})')

# Bone-length sanity check: for a rigid kinematic chain these should be near-constant
# across frames (driven by betas only, not pose). Large std/range here previously
# flagged the per-joint-DLT bug (e.g. joint 4 varied by up to 253mm). model.mano uses
# openpose joint ordering: 0=wrist, 1-4=thumb (CMC..tip), 5-8=index, 9-12=middle,
# 13-16=ring, 17-20=pinky.
for j, name in [(1, 'thumb1'), (4, 'thumb_tip')]:
    mask = joints3d_valid[:, 1]
    lens = np.linalg.norm(joints3d_fused[mask, 1, j] - joints3d_fused[mask, 1, 0], axis=-1) * 1000
    print(f'  right {name} (joint {j}) bone length: mean {lens.mean():.1f}mm, std {lens.std():.1f}mm, '
          f'range {lens.max() - lens.min():.1f}mm')

print('right-hand wrist (joint 0), frames 59-63 (checking for the v1 frame-61 flip):')
for fi in range(59, 64):
    print(f'  frame {frame_keys[fi]}: {np.round(joints3d_fused[fi, 1, 0], 3)}')

In [ ]:
def lerp(a, b, t):
    return a + np.clip(t, 0.0, 1.0) * (b - a)


joints3d_smooth = np.zeros_like(joints3d_fused)
if ENABLE_JOINT_SMOOTHING:
    for si in range(2):
        joints3d_smooth[0, si] = joints3d_fused[0, si]
        for fi in range(1, NUM_FRAMES):
            velocity = np.linalg.norm(joints3d_fused[fi, si] - joints3d_fused[fi - 1, si], axis=-1)  # (21,)
            alpha = lerp(0.85, 0.4, velocity / FAST_MOTION_THRESHOLD)[:, None]  # (21,1)
            joints3d_smooth[fi, si] = lerp(joints3d_smooth[fi - 1, si], joints3d_fused[fi, si], alpha)
else:
    joints3d_smooth[:] = joints3d_fused

print('joints3d_smooth shape:', joints3d_smooth.shape)
print(f'joint-target smoothing: {"ON" if ENABLE_JOINT_SMOOTHING else "OFF"}')
print(f'max |smooth - fused|: {np.abs(joints3d_smooth - joints3d_fused).max() * 1000:.1f}mm')

In [ ]:
IDENTITY_6D = torch.tensor([1., 0., 0., 0., 1., 0.], device=DEVICE)
REFLECT_AXIS = 0  # left-hand reflection trick: mirror world x -> -x

# model.mano (a MANOLayer) feeds hand_pose straight to lbs() as rotation matrices --
# it never adds back model.mano.hand_mean/pose_mean. So hand_pose=IDENTITY_6D means
# the literal flat MANO rest pose, not WiLoR's "neutral" hand. Convert
# model.mano.hand_mean (45,) axis-angle -> per-joint rotmats -> 6D reps, and use that
# as both the warm-start and the pose-prior target instead of IDENTITY_6D.
_hand_mean_rotmats = aa_to_rotmat(model.mano.hand_mean.view(NUM_HAND_JOINTS, 3).to(DEVICE))
POSE_PRIOR_TARGET_6D = torch.cat([_hand_mean_rotmats[:, :, 0], _hand_mean_rotmats[:, :, 1]], dim=-1)


def mano_forward(betas, global_orient_6d, hand_pose_6d):
    global_orient = rot6d_to_rotmat(global_orient_6d.unsqueeze(0)).view(1, 1, 3, 3)
    hand_pose = rot6d_to_rotmat(hand_pose_6d).view(1, NUM_HAND_JOINTS, 3, 3)
    return model.mano(betas=betas.unsqueeze(0), global_orient=global_orient, hand_pose=hand_pose, pose2rot=False)


def fit_mano(target, betas, prev_params, extra_loss_fn=None):
    """Fit global_orient/hand_pose (6D reps) + transl to a (21,3) joint target
    (betas fixed). `prev_params` both warm-starts the optimizer and supplies the
    Step 8 temporal prior; pass None for the first valid frame of a sequence."""
    if prev_params is not None:
        global_orient_6d = prev_params['global_orient_6d'].clone().detach()
        hand_pose_6d = prev_params['hand_pose_6d'].clone().detach()
        transl = prev_params['transl'].clone().detach()
    else:
        global_orient_6d = IDENTITY_6D.clone()
        hand_pose_6d = POSE_PRIOR_TARGET_6D.clone()
        transl = target[0].clone()

    for t in (global_orient_6d, hand_pose_6d, transl):
        t.requires_grad_(True)

    optimizer = torch.optim.Adam([global_orient_6d, hand_pose_6d, transl], lr=LR)
    for _ in range(N_STEPS):
        optimizer.zero_grad()
        out = mano_forward(betas, global_orient_6d, hand_pose_6d)
        pred_joints = out.joints[0] + transl
        loss = ((pred_joints - target) ** 2).sum(dim=-1).mean()
        if ENABLE_POSE_PRIOR:
            loss = loss + POSE_PRIOR_W * ((hand_pose_6d - POSE_PRIOR_TARGET_6D) ** 2).sum()
        if prev_params is not None and ENABLE_TEMPORAL_PRIOR:
            loss = loss + TEMPORAL_W * (
                ((global_orient_6d - prev_params['global_orient_6d']) ** 2).sum()
                + ((hand_pose_6d - prev_params['hand_pose_6d']) ** 2).sum()
                + ((transl - prev_params['transl']) ** 2).sum()
            )
        if extra_loss_fn is not None:
            loss = loss + extra_loss_fn(pred_joints)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        out = mano_forward(betas, global_orient_6d, hand_pose_6d)
        pred_joints = out.joints[0] + transl
        pred_verts = out.vertices[0] + transl
        residual = float(torch.norm(pred_joints - target, dim=-1).mean())

    params = {k: v.detach().clone() for k, v in
              [('global_orient_6d', global_orient_6d), ('hand_pose_6d', hand_pose_6d), ('transl', transl)]}
    return params, pred_joints.detach(), pred_verts.detach(), residual


def reflect(points, side):
    """Map world-space (21,3)/(778,3) points into the optimization space used for
    `side` -- mirrored through x=0 for the left hand (no MANO_LEFT model exists)."""
    if side != 'left_hand':
        return points
    out = points.copy()
    out[:, REFLECT_AXIS] *= -1
    return out


def fit_hand(fi, side, prev):
    si = SIDE_INDEX[side]
    target = torch.tensor(reflect(joints3d_smooth[fi, si], side), dtype=torch.float32, device=DEVICE)
    betas = torch.tensor(beta_fused[fi, si], dtype=torch.float32, device=DEVICE)
    params, pred_joints, pred_verts, residual = fit_mano(target, betas, prev)
    joints_np = reflect(pred_joints.cpu().numpy(), side)
    verts_np = reflect(pred_verts.cpu().numpy(), side)
    return params, joints_np, verts_np, residual


left_verts = np.zeros((NUM_FRAMES, 778, 3), dtype=np.float32)
right_verts = np.zeros((NUM_FRAMES, 778, 3), dtype=np.float32)
left_joints = np.zeros((NUM_FRAMES, 21, 3), dtype=np.float32)
right_joints = np.zeros((NUM_FRAMES, 21, 3), dtype=np.float32)
left_valid = joints3d_valid[:, 0].copy()
right_valid = joints3d_valid[:, 1].copy()
OUT_VERTS = {'left_hand': left_verts, 'right_hand': right_verts}
OUT_JOINTS = {'left_hand': left_joints, 'right_hand': right_joints}

prev_params = {side: None for side in HAND_SIDES}
residuals = {side: [] for side in HAND_SIDES}

t0 = time.time()
for fi in range(NUM_FRAMES):
    fitted_joints, fitted_verts = {}, {}
    for side in HAND_SIDES:
        si = SIDE_INDEX[side]
        if not joints3d_valid[fi, si]:
            prev_params[side] = None
            continue
        params, pj, pv, res = fit_hand(fi, side, prev_params[side])
        prev_params[side] = params
        fitted_joints[side], fitted_verts[side] = pj, pv
        residuals[side].append(res)

    if ENABLE_CONTACT_LOSS and 'left_hand' in fitted_joints and 'right_hand' in fitted_joints:
        wrist_dist = np.linalg.norm(fitted_joints['left_hand'][0] - fitted_joints['right_hand'][0])
        if wrist_dist < CONTACT_THRESHOLD:
            for side, other in [('left_hand', 'right_hand'), ('right_hand', 'left_hand')]:
                other_pts = torch.tensor(reflect(fitted_joints[other], side), dtype=torch.float32, device=DEVICE)

                def repulsion(pred_joints, other_pts=other_pts):
                    d = torch.cdist(pred_joints, other_pts)
                    return CONTACT_W * torch.relu(CONTACT_MIN_DIST - d).pow(2).sum()

                si = SIDE_INDEX[side]
                target = torch.tensor(reflect(joints3d_smooth[fi, si], side), dtype=torch.float32, device=DEVICE)
                betas = torch.tensor(beta_fused[fi, si], dtype=torch.float32, device=DEVICE)
                params, pred_joints, pred_verts, res = fit_mano(target, betas, prev_params[side], extra_loss_fn=repulsion)
                prev_params[side] = params
                fitted_joints[side] = reflect(pred_joints.cpu().numpy(), side)
                fitted_verts[side] = reflect(pred_verts.cpu().numpy(), side)
                residuals[side][-1] = res

    for side in fitted_joints:
        OUT_JOINTS[side][fi] = fitted_joints[side]
        OUT_VERTS[side][fi] = fitted_verts[side]

    if (fi + 1) % 16 == 0 or (fi + 1) == NUM_FRAMES:
        print(f'  {fi + 1}/{NUM_FRAMES} frames ({time.time() - t0:.1f}s)')

np.savez_compressed(
    'final_hand_meshes_wilor_v2.npz',
    left_verts=left_verts, right_verts=right_verts,
    left_valid=left_valid, right_valid=right_valid,
    mano_faces=MANO_FACES, mano_faces_mirrored=MANO_FACES_MIRRORED,
    frame_keys=np.array(frame_keys),
)
print('Saved final_hand_meshes_wilor_v2.npz')
print(f'left valid: {left_valid.sum()}/{NUM_FRAMES}  right valid: {right_valid.sum()}/{NUM_FRAMES}')
for side in HAND_SIDES:
    r = np.array(residuals[side]) * 1000
    print(f'{side}: fit residual mean {r.mean():.1f}mm, max {r.max():.1f}mm')

for side in HAND_SIDES:
    si = SIDE_INDEX[side]
    v = joints3d_valid[:, si]
    err = np.linalg.norm(OUT_JOINTS[side][v, 0] - joints3d_smooth[v, si, 0], axis=-1) * 1000
    print(f'{side}: wrist position error vs joints3d_smooth: mean {err.mean():.2f}mm, max {err.max():.2f}mm')

In [ ]:
import viser
from viser import transforms as tf

with open('final_reconstructed_3d_keypoints.json') as f:
    body_3d = json.load(f)

_hand_data = np.load('final_hand_meshes_wilor_v2.npz', allow_pickle=True)
left_verts = _hand_data['left_verts']
right_verts = _hand_data['right_verts']
left_valid = _hand_data['left_valid']
right_valid = _hand_data['right_valid']

# Calibration world is Y-up; Viser uses Z-up. Flip world Y so the figure is upright.
R_WORLD_TO_VISER = np.array([
    [1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0],
    [0.0, -1.0, 0.0],
], dtype=np.float64)
FRUSTUM_SCALE_BASE = 0.10
FRUSTUM_SCALE_PER_M = 0.03

POSE_CONNECTIONS = [
    (11, 12), (11, 23), (12, 24), (23, 24),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (23, 25), (25, 27), (27, 29), (27, 31), (29, 31),
    (24, 26), (26, 28), (28, 30), (28, 32), (30, 32),
    (0, 1), (1, 2), (2, 3), (0, 4), (4, 5), (5, 6), (3, 7), (6, 8),
    (9, 10),
]
POSE_HAND_JOINTS = {17, 18, 19, 20, 21, 22}


def skeleton_origin(joints_dict):
    """Pelvis midpoint, else mean of all body joints."""
    hip_pts = []
    for hip_id in (23, 24):
        key = str(hip_id)
        if key in joints_dict:
            hip_pts.append([joints_dict[key]['X'], joints_dict[key]['Y'], joints_dict[key]['Z']])
    if len(hip_pts) == 2:
        return np.mean(hip_pts, axis=0)
    body_pts = [[v['X'], v['Y'], v['Z']] for k, v in joints_dict.items() if int(k) < 100]
    if body_pts:
        return np.mean(body_pts, axis=0)
    return np.zeros(3, dtype=np.float64)


def world_to_viser_point(p_world, origin_world):
    return R_WORLD_TO_VISER @ (np.asarray(p_world, dtype=np.float64) - origin_world)


def world_to_viser_pose(T_cw, origin_world):
    T = np.asarray(T_cw, dtype=np.float64).copy()
    T[:3, 3] = R_WORLD_TO_VISER @ (T[:3, 3] - origin_world)
    T[:3, :3] = R_WORLD_TO_VISER @ T[:3, :3]
    wxyz = tf.SO3.from_matrix(T[:3, :3]).wxyz
    return wxyz.astype(np.float64), T[:3, 3].astype(np.float64)


def body_joint_coords(joints_dict, origin_world):
    out = {}
    for j_id, v in joints_dict.items():
        jid = int(j_id)
        if jid >= 100 or jid in POSE_HAND_JOINTS:
            continue
        out[jid] = world_to_viser_point([v['X'], v['Y'], v['Z']], origin_world).astype(np.float32)
    return out


def add_bones(server, name, joint_coords, connections, color):
    segs = []
    for a, b in connections:
        if a in joint_coords and b in joint_coords:
            segs.append([joint_coords[a], joint_coords[b]])
    if segs:
        server.scene.add_line_segments(
            name=name,
            points=np.array(segs, dtype=np.float32),
            colors=np.array(color, dtype=np.uint8),
            line_width=2.0,
        )


def apply_fov(server, fov_deg):
    fov_rad = float(np.deg2rad(fov_deg))
    server.initial_camera.fov = fov_rad
    for client in server.get_clients().values():
        client.camera.fov = fov_rad


def verts_to_viser(verts_world, origin_world):
    return (R_WORLD_TO_VISER @ (verts_world - origin_world).T).T


ORIGIN_WORLD = skeleton_origin(body_3d[frame_keys[0]])

HAND_MESH_SPECS = [
    ('left_hand', left_verts, left_valid, MANO_FACES_MIRRORED, (180, 220, 255), '/hands/left_wilor'),
    ('right_hand', right_verts, right_valid, MANO_FACES, (255, 200, 170), '/hands/right_wilor'),
]


def setup_cameras(server):
    for cam_id, T_cw in CAMERA_WORLD_TRANSFORMS.items():
        calib = CAMERA_CALIB[cam_id]
        K = calib['K']
        fov = 2.0 * np.arctan(calib['height'] / (2.0 * K[1, 1]))
        aspect = float(calib['width'] / calib['height'])
        wxyz, position = world_to_viser_pose(T_cw, ORIGIN_WORLD)
        dist = float(np.linalg.norm(position))
        frustum_scale = float(np.clip(FRUSTUM_SCALE_BASE + FRUSTUM_SCALE_PER_M * dist, 0.08, 0.16))
        server.scene.add_camera_frustum(
            name=f'/cameras/{cam_id}',
            fov=float(fov),
            aspect=aspect,
            scale=frustum_scale,
            line_width=0.5,
            color=(80, 160, 255),
            wxyz=wxyz,
            position=position,
        )
        server.scene.add_frame(
            name=f'/cameras/{cam_id}/axis',
            axes_length=0.08,
            axes_radius=0.004,
            wxyz=wxyz,
            position=position,
        )


def main():
    server = viser.ViserServer()
    running = True
    mesh_handles = {}
    try:
        server.scene.set_up_direction('+z')
        server.scene.configure_environment_map(hdri=None, background=False, environment_intensity=0.6)
        server.gui.configure_theme(dark_mode=True)
        server.scene.set_background_image(np.zeros((8, 8, 3), dtype=np.uint8))

        setup_cameras(server)

        server.initial_camera.position = (2.5, 2.0, 1.6)
        server.initial_camera.look_at = (0.0, 0.0, 0.9)
        server.initial_camera.up = (0.0, 0.0, 1.0)

        print('Viser: open the URL shown above')
        print(f'Origin offset (calibration world): {ORIGIN_WORLD}')

        play_button = server.gui.add_button('Play / Pause')
        stop_button = server.gui.add_button('Stop viewer')
        speed_slider = server.gui.add_slider('Playback Speed (FPS)', min=1, max=60, step=1, initial_value=15)
        frame_slider = server.gui.add_slider('Timeline Frame', min=0, max=NUM_FRAMES - 1, step=1, initial_value=0)
        fov_slider = server.gui.add_slider('Camera FOV (deg)', min=20.0, max=120.0, step=1.0, initial_value=75.0)
        apply_fov(server, fov_slider.value)
        is_playing = False

        @play_button.on_click
        def _(_) -> None:
            nonlocal is_playing
            is_playing = not is_playing

        @stop_button.on_click
        def _stop(_) -> None:
            nonlocal running
            running = False

        @fov_slider.on_update
        def _fov(_) -> None:
            apply_fov(server, fov_slider.value)

        current_idx = 0
        while running:
            if not is_playing:
                current_idx = frame_slider.value
            f_key = frame_keys[current_idx]

            joint_coords = body_joint_coords(body_3d[f_key], ORIGIN_WORLD)
            if joint_coords:
                ids = sorted(joint_coords)
                pts = np.stack([joint_coords[i] for i in ids]).astype(np.float32)
                server.scene.add_point_cloud(
                    name='/skeleton/joints',
                    points=pts,
                    colors=np.full((len(ids), 3), 220, dtype=np.uint8),
                    point_size=0.005,
                )
                add_bones(server, '/skeleton/body', joint_coords, POSE_CONNECTIONS, (0, 255, 120))

            for side, verts, valid, faces, color, name in HAND_MESH_SPECS:
                if valid[current_idx]:
                    v = verts_to_viser(verts[current_idx], ORIGIN_WORLD).astype(np.float32)
                    mesh_handles[name] = server.scene.add_mesh_simple(
                        name=name, vertices=v, faces=faces, color=color, opacity=0.9,
                    )
                elif name in mesh_handles:
                    mesh_handles[name].visible = False

            if is_playing:
                current_idx = (current_idx + 1) % NUM_FRAMES
                frame_slider.value = current_idx
                time.sleep(1.0 / speed_slider.value)
            else:
                time.sleep(0.05)
    except KeyboardInterrupt:
        print('Viewer interrupted.')
    finally:
        server.stop()
        print('Viser stopped.')


main()

In [ ]:
# === Center-camera render ===========================================================
# Renders the fitted MANO meshes (final_hand_meshes_wilor_v2.npz) back onto the
# center camera's own footage -- a quick visual sanity check of the world-frame
# alignment and temporal smoothness of the fit, independent of the Viser viewer.

os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
import pyrender
import trimesh

CENTER_CAM = 'cam03'  # camera whose position is closest to the centroid of all 7
RENDER_SCALE = 0.5    # downscale factor for the output video (speed + file size)
RENDER_FPS = 10
RENDER_OUT_MP4 = 'center_cam_render_v2.mp4'

_hand_data = np.load('final_hand_meshes_wilor_v2.npz', allow_pickle=True)
_left_verts, _right_verts = _hand_data['left_verts'], _hand_data['right_verts']
_left_valid, _right_valid = _hand_data['left_valid'], _hand_data['right_valid']
_mano_faces, _mano_faces_mirrored = _hand_data['mano_faces'], _hand_data['mano_faces_mirrored']

_K = CAMERA_CALIB[CENTER_CAM]['K']
_dist = CAMERA_CALIB[CENTER_CAM]['dist']
_render_w = int(CAMERA_CALIB[CENTER_CAM]['width'] * RENDER_SCALE)
_render_h = int(CAMERA_CALIB[CENTER_CAM]['height'] * RENDER_SCALE)
_K_scaled = _K.copy()
_K_scaled[:2, :] *= RENDER_SCALE

# World -> camera (OpenCV: X right, Y down, Z forward), then a 180deg rotation about X
# (MIRROR_GL) into pyrender/OpenGL's view space (X right, Y up, Z backward).
_T_wc = np.linalg.inv(CAMERA_WORLD_TRANSFORMS[CENTER_CAM])
_R_wc, _t_wc = _T_wc[:3, :3], _T_wc[:3, 3]
_MIRROR_GL = np.array([1.0, -1.0, -1.0])

_render_camera = pyrender.IntrinsicsCamera(fx=_K_scaled[0, 0], fy=_K_scaled[1, 1],
                                            cx=_K_scaled[0, 2], cy=_K_scaled[1, 2],
                                            znear=0.01, zfar=10.0)
_offrenderer = pyrender.OffscreenRenderer(viewport_width=_render_w, viewport_height=_render_h)
_writer = cv2.VideoWriter(RENDER_OUT_MP4, cv2.VideoWriter_fourcc(*'mp4v'), RENDER_FPS,
                           (_render_w, _render_h))

t0 = time.time()
with h5py.File(H5_PATH, 'r') as f5:
    for fi, frame_key in enumerate(frame_keys):
        img = decode_frame(f5, CENTER_CAM, frame_key)
        undist = cv2.undistort(img, _K, _dist)
        bg = cv2.resize(undist, (_render_w, _render_h)).astype(np.float32) / 255.0

        scene = pyrender.Scene(bg_color=[0, 0, 0, 0], ambient_light=(0.6, 0.6, 0.6))
        for verts_all, valid_all_f, faces, color in [
            (_left_verts, _left_valid, _mano_faces_mirrored, (0.71, 0.86, 1.0)),
            (_right_verts, _right_valid, _mano_faces, (1.0, 0.78, 0.67)),
        ]:
            if not valid_all_f[fi]:
                continue
            verts_cam = (_R_wc @ verts_all[fi].T).T + _t_wc
            verts_gl = verts_cam * _MIRROR_GL
            material = pyrender.MetallicRoughnessMaterial(metallicFactor=0.0, alphaMode='OPAQUE',
                                                            baseColorFactor=(*color, 1.0))
            tm = trimesh.Trimesh(verts_gl, faces, process=False)
            scene.add(pyrender.Mesh.from_trimesh(tm, material=material, smooth=True))
        scene.add(_render_camera, pose=np.eye(4))

        color_rgba, _ = _offrenderer.render(scene, flags=pyrender.RenderFlags.RGBA)
        color_rgba = color_rgba.astype(np.float32) / 255.0
        alpha = color_rgba[:, :, 3:4]
        color_bgr = color_rgba[:, :, :3][:, :, ::-1]
        composite = color_bgr * alpha + bg * (1 - alpha)
        _writer.write((composite * 255).astype(np.uint8))

        if (fi + 1) % 16 == 0 or (fi + 1) == NUM_FRAMES:
            print(f'  {fi + 1}/{NUM_FRAMES} frames ({time.time() - t0:.1f}s)')

_writer.release()
_offrenderer.delete()
print(f'Saved {RENDER_OUT_MP4} ({_render_w}x{_render_h} @ {RENDER_FPS}fps, camera={CENTER_CAM})')

In [ ]:
# === Center-camera render: fitted vs. Simple AVG Debug, side by side =================
# Same center-camera overlay as the previous cell, rendering both
# final_hand_meshes_wilor_v2.npz (fitted pipeline) and simple_avg_debug_v2.npz (Simple
# AVG Debug) onto the same footage and placing them side by side for direct comparison.

os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')
import pyrender
import trimesh

RENDER_COMPARE_OUT_MP4 = 'center_cam_render_compare_v2.mp4'

_fitted = np.load('final_hand_meshes_wilor_v2.npz', allow_pickle=True)
_simpleavg = np.load('simple_avg_debug_v2.npz', allow_pickle=True)
_mano_faces, _mano_faces_mirrored = _fitted['mano_faces'], _fitted['mano_faces_mirrored']

_K = CAMERA_CALIB[CENTER_CAM]['K']
_dist = CAMERA_CALIB[CENTER_CAM]['dist']
_render_w = int(CAMERA_CALIB[CENTER_CAM]['width'] * RENDER_SCALE)
_render_h = int(CAMERA_CALIB[CENTER_CAM]['height'] * RENDER_SCALE)
_K_scaled = _K.copy()
_K_scaled[:2, :] *= RENDER_SCALE

_T_wc = np.linalg.inv(CAMERA_WORLD_TRANSFORMS[CENTER_CAM])
_R_wc, _t_wc = _T_wc[:3, :3], _T_wc[:3, 3]
_MIRROR_GL = np.array([1.0, -1.0, -1.0])

_render_camera = pyrender.IntrinsicsCamera(fx=_K_scaled[0, 0], fy=_K_scaled[1, 1],
                                            cx=_K_scaled[0, 2], cy=_K_scaled[1, 2],
                                            znear=0.01, zfar=10.0)
_offrenderer = pyrender.OffscreenRenderer(viewport_width=_render_w, viewport_height=_render_h)
_writer = cv2.VideoWriter(RENDER_COMPARE_OUT_MP4, cv2.VideoWriter_fourcc(*'mp4v'), RENDER_FPS,
                           (_render_w * 2, _render_h))


def _render_overlay(bg, fi, hand_data):
    left_verts, right_verts = hand_data['left_verts'], hand_data['right_verts']
    left_valid, right_valid = hand_data['left_valid'], hand_data['right_valid']
    scene = pyrender.Scene(bg_color=[0, 0, 0, 0], ambient_light=(0.6, 0.6, 0.6))
    for verts_all, valid_all_f, faces, color in [
        (left_verts, left_valid, _mano_faces_mirrored, (0.71, 0.86, 1.0)),
        (right_verts, right_valid, _mano_faces, (1.0, 0.78, 0.67)),
    ]:
        if not valid_all_f[fi]:
            continue
        verts_cam = (_R_wc @ verts_all[fi].T).T + _t_wc
        verts_gl = verts_cam * _MIRROR_GL
        material = pyrender.MetallicRoughnessMaterial(metallicFactor=0.0, alphaMode='OPAQUE',
                                                        baseColorFactor=(*color, 1.0))
        tm = trimesh.Trimesh(verts_gl, faces, process=False)
        scene.add(pyrender.Mesh.from_trimesh(tm, material=material, smooth=True))
    scene.add(_render_camera, pose=np.eye(4))

    color_rgba, _ = _offrenderer.render(scene, flags=pyrender.RenderFlags.RGBA)
    color_rgba = color_rgba.astype(np.float32) / 255.0
    alpha = color_rgba[:, :, 3:4]
    color_bgr = color_rgba[:, :, :3][:, :, ::-1]
    composite = color_bgr * alpha + bg * (1 - alpha)
    return (composite * 255).astype(np.uint8)


t0 = time.time()
with h5py.File(H5_PATH, 'r') as f5:
    for fi, frame_key in enumerate(frame_keys):
        img = decode_frame(f5, CENTER_CAM, frame_key)
        undist = cv2.undistort(img, _K, _dist)
        bg = cv2.resize(undist, (_render_w, _render_h)).astype(np.float32) / 255.0

        left_frame = _render_overlay(bg, fi, _fitted)
        right_frame = _render_overlay(bg, fi, _simpleavg)
        cv2.putText(left_frame, 'Fitted', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        cv2.putText(right_frame, 'Simple AVG', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        combined = cv2.hconcat([left_frame, right_frame])
        _writer.write(combined)

        if (fi + 1) % 16 == 0 or (fi + 1) == NUM_FRAMES:
            print(f'  {fi + 1}/{NUM_FRAMES} frames ({time.time() - t0:.1f}s)')

_writer.release()
_offrenderer.delete()
print(f'Saved {RENDER_COMPARE_OUT_MP4} ({_render_w * 2}x{_render_h} @ {RENDER_FPS}fps, camera={CENTER_CAM})')

In [ ]:
# Optional: stop a Viser server if the viewer cell did not shut down cleanly.
# Re-run the viewer cell (cell above) after stopping any old server.
print('Use Stop viewer in Viser GUI, or interrupt the cell, to stop the viewer.')